# Pipeline OCR tiếng Việt (viết ngang, trái sang phải) bằng Pytesseract (detection) + Qwen3-VL-4B-Instruct (recognition)

## 1. Kiểm tra môi trường GPU & Cài đặt thư viện phụ thuộc
> **Mô tả:**
> - Kiểm tra phần cứng GPU (`nvidia-smi`) và phiên bản Python.
> - Tiến hành cài đặt tự động các thư viện cần thiết như `PyMuPDF` (xử lý PDF), `pytesseract`(Detection).


In [1]:
import sys, os
!nvidia-smi
print("\nPython:", sys.version)
print("CWD:", os.getcwd())

Tue Jul 21 17:02:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!sudo apt-get update -qq && sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-vie zstd
!pip install -q pytesseract PyMuPDF Pillow opencv-python-headless matplotlib tqdm requests ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (

## 2. Cấu hình đường dẫn & tham số chạy Pipeline
> **Mô tả:**
> - Khai báo thông tin tài liệu (`WORK_ID`, `FILE_PATH`) và **loại file pdf (`FILE_TYPE`)**:
>   - `pdf_scan`: Cần render ảnh và gửi qua VLM model để OCR.
>   - `pdf_text`: Đã có sẵn lớp văn bản, chỉ cần rút trích và chuẩn hóa.
> - Đối với `pdf_text`, có thể thiết lập phạm vi trang cần xử lý (`PAGE_START`, `PAGE_COUNT`), thông số render DPI (300 DPI), thư mục đầu ra của ảnh từng trang (`/kaggle/working/pages`)
> - Tạo thư mục đầu ra chứa kết quả là văn bản thô đã được xử lý bởi pipeline này (`/kaggle/working/output`)


In [6]:
from pathlib import Path

WORK_ID    = "HVB_003"
WORK_TITLE = "HVB_003_TITLE"

# Loại file & đường dẫn ("pdf_scan", "pdf_text")
FILE_TYPE = "pdf_scan"
FILE_PATH = "/kaggle/input/datasets/james2072/hvb-vie/vie/an-nam-chi-nguyen/HVB_002_PDFScan_Viet_An Nam Chí Nguyên.pdf"

PDF_PATH = FILE_PATH  # Giữ cho tương thích với code cũ
PDF_RENDER_DPI = 300

# Trang bắt đầu (0-indexed) và số trang cần OCR
PAGE_START = 281       # bắt đầu từ trang nào (0-indexed)
PAGE_COUNT = 1    # OCR bao nhiêu trang (None = tất cả từ PAGE_START)

# Tesseract config
TESS_LANG = "vie"
TESS_PSM  = 6        # 6 = single block, 3 = auto, 4 = single column
TESS_OEM  = 3        # 3 = LSTM engine

# LLM Corrector (Ollama)
USE_LLM_CORRECTOR = True
LLM_MODEL_NAME = "qwen3-vl-4b"

# ======================== ĐƯỜNG DẪN ========================
WORK_DIR     = Path("/kaggle/working")
PAGES_DIR    = WORK_DIR / "pages"
OUTPUT_DIR   = WORK_DIR / "output"
OUTPUT_FILE  = OUTPUT_DIR / f"{WORK_ID}_vie_raw.txt"

for d in [PAGES_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"WORK_ID:    {WORK_ID}")
print(f"WORK_TITLE: {WORK_TITLE}")
print(f"FILE_PATH:  {FILE_PATH}")
print(f"FILE_TYPE:  {FILE_TYPE}")
print(f"OCR range:  start={PAGE_START}, count={PAGE_COUNT or 'ALL'}")
print(f"Tesseract:  lang={TESS_LANG}, psm={TESS_PSM}, oem={TESS_OEM}")
print(f"LLM:        {'ON' if USE_LLM_CORRECTOR else 'OFF'} ({LLM_MODEL_NAME})")
print(f"OUTPUT:     {OUTPUT_FILE}")

WORK_ID:    HVB_003
WORK_TITLE: HVB_003_TITLE
FILE_PATH:  /kaggle/input/datasets/james2072/hvb-vie/vie/an-nam-chi-nguyen/HVB_002_PDFScan_Viet_An Nam Chí Nguyên.pdf
FILE_TYPE:  pdf_scan
OCR range:  start=281, count=1
Tesseract:  lang=vie, psm=6, oem=3
LLM:        ON (qwen3-vl-4b)
OUTPUT:     /kaggle/working/output/HVB_003_vie_raw.txt


## 3. Cấu hình & Khởi động Local VLM Server (Qwen3-VL-4B)
> **Mô tả:**
> - Nếu `FILE_TYPE == "pdf_text"`: Bỏ qua khởi động VLM server để tiết kiệm tài nguyên GPU.
> - Nếu `FILE_TYPE == "pdf_scan"` & `USE_LLM_CORRECTOR = True`: Tự động clone repository `qwen3-vl-4b-hosting`, khởi chạy server API ở dạng background process (port 8000), và kiểm tra `/health` cho tới khi mô hình VLM sẵn sàng nhận request.

In [4]:
import subprocess, os

if FILE_TYPE == "pdf_text":
    print(f"⏭ FILE_TYPE='{FILE_TYPE}' (đã có text layer sẵn) — Bỏ qua VLM server.")
elif USE_LLM_CORRECTOR:
    REPO_URL = "https://github.com/tqth/qwen3-vl-4b-hosting.git"
    REPO_DIR = "/kaggle/working/qwen3-vl-4b-hosting"

    if not os.path.exists(REPO_DIR):
        print("Clone repo VLM server ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print("Repo đã có sẵn, bỏ qua clone.")

    # Khởi động server
    import time, requests

    log_path = "/kaggle/working/start_server.log"
    proc = subprocess.Popen(
        ["python", "start_server.py", "--model", "qwen3vl"],
        cwd=REPO_DIR,
        stdout=open(log_path, "w"),
        stderr=subprocess.STDOUT,
    )
    print(f"Đã khởi động (PID={proc.pid}), đợi server load model ...")

    for i in range(60):
        try:
            r = requests.get("http://localhost:8000/health", timeout=3)
            if r.status_code == 200:
                print("✓ Server sẵn sàng:", r.json())
                break
        except Exception:
            pass
        time.sleep(15)
        print(f"  ...{(i+1)*15}s, xem log: {log_path}")
    else:
        print("⚠ Timeout, xem log để debug:")
        print(open(log_path).read()[-3000:])
else:
    print("⏭ LLM Corrector TẮT — bỏ qua VLM server")

Clone repo VLM server ...


Cloning into '/kaggle/working/qwen3-vl-4b-hosting'...


Đã khởi động (PID=910), đợi server load model ...
  ...15s, xem log: /kaggle/working/start_server.log
  ...30s, xem log: /kaggle/working/start_server.log
  ...45s, xem log: /kaggle/working/start_server.log
  ...60s, xem log: /kaggle/working/start_server.log
✓ Server sẵn sàng: {'model': 'Qwen/Qwen3-VL-4B-Instruct', 'status': 'ok'}


## 4. Render các trang PDF Scan thành ảnh chất lượng cao (PyMuPDF)
> **Mô tả:**
> - Nếu `FILE_TYPE == "pdf_text"`: Bỏ qua bước tách ảnh vì không cần OCR.
> - Nếu `FILE_TYPE == "pdf_scan"`: Mở file PDF bằng `PyMuPDF` (`fitz`), cắt/render từng trang theo dải trang cấu hình (`PAGE_START` đến `PAGE_COUNT`) thành file ảnh PNG với độ phân giải 300 DPI để phục vụ nhận diện chữ tiếng Việt sắc nét.

In [7]:
import fitz  # PyMuPDF
from pathlib import Path

# ===================== INPUT =====================
PDF_PATH = Path(FILE_PATH)

# ===================== OUTPUT ====================
PAGES_DIR = Path("/kaggle/working/pages")
PAGES_DIR.mkdir(parents=True, exist_ok=True)

if FILE_TYPE == "pdf_text":
    print(f"⏭ FILE_TYPE='{FILE_TYPE}' (text có sẵn) — Bỏ qua render ảnh.")
else:
    # Xóa ảnh cũ nếu có
    for f in PAGES_DIR.glob("*.png"):
        f.unlink()

    # Lấy tên file làm prefix
    WORK_ID = PDF_PATH.stem.split("_PDFScan")[0]   # HVB_005

    # ===================== RENDER ====================
    doc = fitz.open(PDF_PATH)

    DPI = PDF_RENDER_DPI
    zoom = DPI / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    print(f"Tổng số trang: {len(doc)}")

    start_idx = (PAGE_START - 1) if PAGE_START else 0
    end_idx = len(doc) if PAGE_COUNT is None else min(start_idx + PAGE_COUNT, len(doc))

    for i in range(start_idx, end_idx):
        page = doc[i]

        pix = page.get_pixmap(
            matrix=matrix,
            alpha=False
        )

        output_path = PAGES_DIR / f"{WORK_ID}_p{i+1:04d}.png"

        pix.save(output_path)

        print(f"Đã lưu: {output_path.name}")

    doc.close()
    print(f"\nHoàn tất! Ảnh được lưu tại: {PAGES_DIR}")

Tổng số trang: 582
Đã lưu: HVB_002_p0281.png

Hoàn tất! Ảnh được lưu tại: /kaggle/working/pages


## 5. Khai triển các hàm hỗ trợ đọc File & Làm sạch văn bản
> **Mô tả:**
> - `load_and_process_input`: Hàm xử lý phân loại đầu vào (`pdf_text` rút trích chữ NFC, `pdf_scan` đọc ảnh).
> - `_is_noise_line`: Thuật toán lọc nhiễu phát hiện các dòng rác (số trang, ký tự vô nghĩa, lặp từ nhiễu OCR).
> - `filter_for_alignment`: Hàm làm sạch văn bản tiếng Việt sau OCR (xóa ký tự cước chú dạng superscript `¹²³`, loại bỏ ô vuông/ký tự lạ, chuẩn hóa khoảng trắng) phục vụ cho Sentence Alignment.


In [8]:
import os, fitz, cv2, unicodedata, re
import numpy as np

def load_and_process_input(file_path, file_type, work_id=None):
    """
    Đọc và xử lý file PDF đầu vào theo 2 file_type:
    - pdf_text: Đọc trực tiếp text layer từ file PDF bằng PyMuPDF
    - pdf_scan: Render các trang PDF thành ảnh (300 DPI) để chạy OCR
    """
    abs_path = str(file_path)
    if not os.path.exists(abs_path):
        print(f"Không tìm thấy file: {abs_path}")
        return [], "unknown"
    print(f"  -> Đang đọc: {os.path.basename(abs_path)}")
    if file_type == "pdf_text":
        doc = fitz.open(abs_path)
        text_pages = []
        for page in doc:
            raw_page = page.get_text("text")
            text_pages.append(unicodedata.normalize("NFC", raw_page))
        doc.close()
        return text_pages, "text"
    elif file_type == "pdf_scan":
        doc = fitz.open(abs_path)
        images = []
        mat = fitz.Matrix(300 / 72, 300 / 72)  # Render 300 DPI sắc nét
        for page in doc:
            pix = page.get_pixmap(matrix=mat)
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
            if pix.n == 4:
                img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            images.append(img)
        doc.close()
        return images, "image"
    return [], "unknown"


def _is_noise_line(s):
    """Kiểm tra dòng có phải rác (số trang, ký tự lạc, nhiễu OCR) không."""
    if not s or len(s) <= 1:
        return True
    if re.fullmatch(r'[\d\s/.,\-–\u2014:]+', s):
        return True
    letters = re.findall(r'[a-zA-Z\u00c0-\u024f\u1e00-\u1eff]', s)
    if len(letters) < 2:
        return True
    words = s.split()
    if len(words) >= 6:
        from collections import Counter
        most_common_count = Counter(words).most_common(1)[0][1]
        if most_common_count / len(words) >= 0.5:
            return True
    return False

def filter_for_alignment(text):
    """
    Lọc sạch văn bản tiếng Việt sau OCR để phục vụ Sentence Alignment:
    - Xóa chỉ số cước chú (phần¹ → phần)
    - Loại bỏ ký tự rác/nhiễu OCR
    - Chuẩn hóa khoảng trắng
    - Loại bỏ dòng trống/rác
    """
    if not text:
        return ""
    lines = text.split("\n")
    inline_citation_regex = re.compile(
        r"[¹²³⁴⁵⁶⁷⁸⁹⁰†‡]+|(?<=[a-zA-Z\u00c0-\u024f\u1e00-\u1eff])[\(\[\{]\d+[\)\]\}]"
    )
    strict_vietnamese_regex = re.compile(
        r"[^a-zA-Z0-9\u00c0-\u024f\u1e00-\u1eff\u0300-\u036f\s\.,:;?!\-–—\(\)\[\]\"'""''/]",
        flags=re.UNICODE
    )
    clean_lines = []
    for line in lines:
        line_str = line.strip()
        if not line_str:
            continue
        line_str = inline_citation_regex.sub("", line_str)
        line_str = strict_vietnamese_regex.sub(" ", line_str)
        line_str = re.sub(r"[\(\[\{]\s*[\)\]\}]", "", line_str)
        line_str = re.sub(r"\s+", " ", line_str).strip()
        if line_str and not _is_noise_line(line_str):
            clean_lines.append(line_str)
    return "\n".join(clean_lines)

print("✓ Loaded: load_and_process_input, filter_for_alignment")

✓ Loaded: load_and_process_input, filter_for_alignment


## 6. Pipeline tổng — Xử lý nhận diện & Trích xuất lưu file
> **Mô tả:**
> - **Nếu `FILE_TYPE == "pdf_text"`**: Đọc trực tiếp text layer qua `load_and_process_input`, lọc sạch nhiễu qua `filter_for_alignment`, và ghi thẳng ra file `OUTPUT_FILE`.
> - **Nếu `FILE_TYPE == "pdf_scan"`**: Duyệt qua danh sách ảnh trang đã render, mã hóa Base64 gửi lên VLM Server (Qwen3-VL-4B) kèm prompt chuyên biệt cho sách Quốc ngữ cổ, thực hiện lọc sạch text và tự động lưu **Checkpoint định kỳ 10 trang/lần** ra file `OUTPUT_FILE`.

In [9]:
if FILE_TYPE == "pdf_text":
    # Đã có text layer sẵn — đọc thẳng + lọc rác cước chú/viền trang, không cần OCR
    pages, _ = load_and_process_input(FILE_PATH, FILE_TYPE, WORK_ID)
    full_text = "\n".join(pages)
    full_text = filter_for_alignment(full_text)
    OUTPUT_FILE.write_text(full_text.strip() + "\n\n", encoding="utf-8")
    print(f"  → Bỏ qua OCR (text có sẵn), đã chuẩn hóa & lưu vào file: {OUTPUT_FILE} ({len(full_text):,} ký tự)")
else:
    import requests, base64, re, io
    from tqdm.auto import tqdm

    SERVER_URL = "http://localhost:8000"

    VIE_OCR_PROMPT = """You are a high-accuracy OCR engine for scanned historical Vietnamese (Quốc ngữ) books.

Your task is to transcribe the page exactly as it appears.

Rules:

1. Transcribe ALL visible Vietnamese text.
2. Preserve the original spelling, punctuation, capitalization, accents, line breaks, and historical orthography.
3. Read the page from top to bottom and left to right.
4. Do NOT modernize or normalize the text.
5. Do NOT correct OCR errors by guessing.
6. If one or more characters cannot be recognized confidently, replace only those characters with '?'.
7. Preserve page numbers, headings, titles, and short isolated lines if they are part of the printed page.
8. Ignore only obvious scanning artifacts such as stains, shadows, page borders, folds, and background noise.
9. Do NOT translate, summarize, explain, or annotate.
10. Do NOT output Markdown, code fences, quotation marks, or any extra text.

Output ONLY the transcription.
"""

    def pil_image_to_base64(img_pil):
        max_dim = 1536

        w, h = img_pil.size

        if max(w, h) > max_dim:
            scale = max_dim / max(w, h)
            img_pil = img_pil.resize(
                (int(w * scale), int(h * scale)),
                Image.Resampling.LANCZOS
            )

        buffer = io.BytesIO()
        img_pil.save(buffer, format="PNG")

        return base64.b64encode(buffer.getvalue()).decode()


    import time
    import requests
    from requests.exceptions import ReadTimeout, ConnectionError

    def recognize_page_vlm(img_pil, timeout=900, retries=4, retry_wait=10):

        img_b64 = pil_image_to_base64(img_pil)

        payload = {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": VIE_OCR_PROMPT
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{img_b64}"
                            }
                        }
                    ]
                }
            ],
            "temperature": 0.0,
            "max_tokens": 4096
        }

        last_error = None

        for attempt in range(1, retries + 1):
            try:
                print(f"  OCR attempt {attempt}/{retries}")

                r = requests.post(
                    f"{SERVER_URL}/v1/chat/completions",
                    json=payload,
                    timeout=timeout
                )

                r.raise_for_status()

                text = r.json()["choices"][0]["message"]["content"]

                text = re.sub(
                    r"^```[a-zA-Z]*\n?(.*?)\n?```$",
                    r"\1",
                    text,
                    flags=re.DOTALL
                ).strip()

                return text

            except (ReadTimeout, ConnectionError) as e:
                last_error = e

                if attempt == retries:
                    break

                print(f"  ⚠️ Timeout, sẽ thử lại sau {retry_wait}s...")
                time.sleep(retry_wait)

        raise last_error


    # ================= OCR =================
    from pathlib import Path
    from PIL import Image
    PAGES_DIR = Path("/kaggle/working/pages")
    pages = sorted(PAGES_DIR.glob("*.png"))
    all_texts = []

    for i, page_path in enumerate(tqdm(pages, desc=f"VLM OCR Việt {WORK_ID}")):
        page_name = Path(page_path).stem
        print(f"\n--- Trang {page_name} ---")

        img = Image.open(page_path).convert("RGB")

        try:
            raw_text = recognize_page_vlm(img)
            final_text = filter_for_alignment(raw_text)

        except Exception as e:
            print(f"❌ Bỏ qua {page_name}: {e}")
            final_text = f"\n===== OCR FAILED: {page_name} =====\n"

        all_texts.append(final_text)

        if (i + 1) % 10 == 0:
            temp_text = "\n\n".join(t.strip() for t in all_texts if t.strip())
            OUTPUT_FILE.write_text(temp_text, encoding="utf-8")
            print(f"💾 Checkpoint: {i + 1} trang")

    if all_texts:
        full_text = "\n\n".join(t.strip() for t in all_texts if t.strip())
        OUTPUT_FILE.write_text(full_text, encoding="utf-8")
        
        total_pages = len(all_texts)
        leftover = total_pages % 10
        
        if leftover > 0:
            print(f"\n💾 [Checkpoint Cuối] Đã lưu bù {leftover} trang lẻ cuối cùng.")
            
        print(f"\n✅ Đã hoàn tất xử lý và lưu tổng cộng {total_pages} trang vào: {OUTPUT_FILE}")
        print(f"Độ dài text: {len(full_text):,} ký tự")

VLM OCR Việt HVB_002:   0%|          | 0/1 [00:00<?, ?it/s]


--- Trang HVB_002_p0281 ---
  OCR attempt 1/4

💾 [Checkpoint Cuối] Đã lưu bù 1 trang lẻ cuối cùng.

✅ Đã hoàn tất xử lý và lưu tổng cộng 1 trang vào: /kaggle/working/output/HVB_003_vie_raw.txt
Độ dài text: 1,740 ký tự
